# Llama-3.2-3B Base — Answers Without Context

What does the **raw, un-adapted** `meta-llama/Llama-3.2-3B` checkpoint answer when given
only the question from the test split — no context, no CPT, no QA fine-tuning? This
measures the base model's own parametric Sinhala history knowledge, as a reference point
against the context-grounded results in `llama-scripts/qa-evaluation.ipynb`.

Adapted from `qa-evaluation.ipynb`: same test-split loading and scoring helpers, but the
context section is dropped from the prompt entirely (there is nothing to ground against),
so there is no evidence-window retrieval and no grounding gate — the model's raw generation
is the final answer.

In [ ]:
# Installs the packages needed to load and run the model.
%uv pip install -q "transformers>=4.51,<5" accelerate safetensors huggingface_hub hf_transfer

In [ ]:
# Configuration.
import json
import os
import re
import unicodedata
from collections import Counter
from pathlib import Path

import torch

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# meta-llama/Llama-3.2-3B is gated — accept the license at huggingface.co/meta-llama/Llama-3.2-3B
# and use an HF_TOKEN with access. unsloth/Llama-3.2-3B is an ungated mirror with an
# identical tokenizer/weights if the gate is a blocker.
MODEL_ID = "meta-llama/Llama-3.2-3B"

TEST_CANDIDATES = [
    Path("/tmp/test_updated.jsonl"),
    Path("/tmp/test.jsonl"),
    Path("new_split_v2/test.jsonl"),
    Path("../new_split_v2/test.jsonl"),
]
TEST_PATH = next((p for p in TEST_CANDIDATES if p.is_file()), TEST_CANDIDATES[0])

_slug = re.sub(r"[^a-z0-9]+", "-", MODEL_ID.lower()).strip("-")
RESULTS_JSONL = Path(f"/tmp/{_slug}-nocontext-eval.jsonl")
RESULTS_TXT = Path(f"/tmp/{_slug}-nocontext-results.txt")

NO_ANSWER = "මෙම ප්‍රශ්නයට පිළිතුරු දීමට ප්‍රමාණවත් තොරතුරු නොමැත."
MAX_NEW_TOKENS = 350         # audited against gold-answer lengths under THIS model's stock
                              # tokenizer (measured max 313 tokens — far more than the 48 used
                              # elsewhere in this project, since this tokenizer has no
                              # Sinhala-specific vocabulary); the cell below re-checks this
REPETITION_PENALTY = 1.05

print("Model     :", MODEL_ID)
print("Test file :", TEST_PATH, "| exists:", TEST_PATH.is_file())

In [ ]:
# Logs in to Hugging Face — meta-llama/Llama-3.2-3B is gated, so this needs an HF_TOKEN with
# access to it. Reads the token from the environment; never paste a token into a cell.
from huggingface_hub import login

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("Logged in to Hugging Face from HF_TOKEN.")
else:
    print("No HF_TOKEN in the environment — loading a gated model will fail.")

In [ ]:
# Loads the base model and tokenizer (no adapters, no merges — the checkpoint as published).
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=model_dtype,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded.  vocab:", len(tokenizer), "| dtype:", model_dtype, "| device:", model.device)
print("bos/eos/pad:", tokenizer.bos_token_id, "/", tokenizer.eos_token_id, "/", tokenizer.pad_token_id)

In [ ]:
# Text cleaning, the no-context prompt, and data loading — adapted from qa-evaluation.ipynb.
# INSTRUCTION no longer mentions a context (there isn't one); everything else (word
# tokenization, stopword list, answer normalization) is unchanged since it's context-agnostic.
INSTRUCTION = "උපදෙස්: පහත ප්‍රශ්නයට කෙටියෙන් හා නිවැරදිව, එක් පේළියකින් පිළිතුරු දෙන්න."

SINHALA_WORD_RE = re.compile(r"[\w඀-෿]+", re.UNICODE)
STOPWORDS = {
    "හා", "සහ", "හෝ", "දී", "ද", "ය", "යි", "වේ", "විය", "වූ", "ලෙස",
    "විසින්", "සඳහා", "සිට", "දක්වා", "එම", "මෙම", "ඒ", "ඔහු", "ඇය",
    "කුමක්ද", "කවුද", "කවදාද", "කෙසේද", "කොපමණද", "මොනවාද",
}


def clean_text(value):
    text = unicodedata.normalize("NFC", str(value or ""))
    return text.replace("\r\n", "\n").replace("\r", "\n").strip()


def lexical_tokens(value):
    tokens = [token.casefold() for token in SINHALA_WORD_RE.findall(clean_text(value))]
    return [token for token in tokens if len(token) >= 2 and token not in STOPWORDS]


def build_prompt(question):
    return f"{INSTRUCTION}\n\nප්‍රශ්නය:\n{clean_text(question)}\n\nපිළිතුර:\n"


def canonical_answer(item):
    if item.get("answerable") is False:
        return NO_ANSWER
    answer = clean_text(item.get("answer", ""))
    return answer if answer else NO_ANSWER


def load_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Required JSONL file not found: {path}")

    records = []
    fingerprints = set()
    dropped = duplicates = 0

    with path.open("r", encoding="utf-8-sig") as handle:
        for line_number, line in enumerate(handle, 1):
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"{path}:{line_number}: {error}") from error

            question = clean_text(item.get("question"))
            answerable = item.get("answerable")
            if type(answerable) is not bool:
                answerable = bool(clean_text(item.get("answer")))

            normalized = {
                "question": question,
                "answer": clean_text(item.get("answer")),
                "answerable": answerable,
                "grade": item.get("grade"),
                "chapter": item.get("chapter"),
            }
            if not question or (answerable and not normalized["answer"]):
                dropped += 1
                continue

            fingerprint = (normalized["question"], normalized["answer"], normalized["answerable"])
            if fingerprint in fingerprints:
                duplicates += 1
                continue
            fingerprints.add(fingerprint)
            records.append(normalized)

    return records, dropped, duplicates


print("Prompt and data helpers ready (no-context).")

In [ ]:
# Generation (no windowing, no grounding — there is no context to window or ground against)
# and scoring, adapted from qa-evaluation.ipynb.
def run_qa(question):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True).to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=REPETITION_PENALTY,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    return answer.splitlines()[0].strip(" []{}()<>\"'`") if answer else ""


def normalize_answer(value):
    text = clean_text(value).casefold()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\r\n[]{}()<>\"'`.,!?;:।෴")


def is_no_answer(value):
    normalized = normalize_answer(value)
    return normalized == normalize_answer(NO_ANSWER) or "ප්‍රමාණවත් තොරතුරු නොමැත" in normalized


def token_f1(prediction, reference):
    prediction_digits = re.findall(r"\d+", normalize_answer(prediction))
    reference_digits = re.findall(r"\d+", normalize_answer(reference))
    if reference_digits and prediction_digits != reference_digits:
        return 0.0
    prediction_tokens = lexical_tokens(prediction)
    reference_tokens = lexical_tokens(reference)
    if not prediction_tokens and not reference_tokens:
        return 1.0
    if not prediction_tokens or not reference_tokens:
        return 0.0
    common = Counter(prediction_tokens) & Counter(reference_tokens)
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(prediction_tokens)
    recall = overlap / len(reference_tokens)
    return 2 * precision * recall / (precision + recall)


print("run_qa(question) and scoring helpers ready.")

In [ ]:
# Loads the test split and checks MAX_NEW_TOKENS is enough for the gold answers under this
# model's own (stock, less Sinhala-efficient) tokenizer — raise it if the warning below fires.
test_records, dropped_rows, duplicate_rows = load_jsonl(TEST_PATH)
print(f"Loaded {len(test_records)} unique records from {TEST_PATH}")
print(f"Dropped invalid/empty: {dropped_rows}; exact duplicates removed: {duplicate_rows}")
print(Counter(r["answerable"] for r in test_records))

gold_lengths = sorted(
    len(tokenizer(canonical_answer(r), add_special_tokens=False)["input_ids"]) for r in test_records
)
print(
    f"\nGold answer tokens under this tokenizer: median {gold_lengths[len(gold_lengths)//2]}, "
    f"p95 {gold_lengths[int(len(gold_lengths)*0.95)]}, max {gold_lengths[-1]} "
    f"(MAX_NEW_TOKENS={MAX_NEW_TOKENS})"
)
if gold_lengths[-1] > MAX_NEW_TOKENS:
    print("WARNING: raise MAX_NEW_TOKENS above — the longest gold answer does not fit.")

In [ ]:
# ---- Full run: generate an answer for every question, no context given ----
predictions = []
transcript_lines = []
exact_correct = 0
f1_total = 0.0

with RESULTS_JSONL.open("w", encoding="utf-8", newline="\n") as results_file:
    for index, item in enumerate(test_records, 1):
        reference = canonical_answer(item)
        prediction = run_qa(item["question"])

        exact = normalize_answer(prediction) == normalize_answer(reference)
        f1 = token_f1(prediction, reference)
        exact_correct += int(exact)
        f1_total += f1

        results_file.write(json.dumps({
            "index": index,
            "grade": item.get("grade"),
            "chapter": item.get("chapter"),
            "question": item["question"],
            "reference": reference,
            "prediction": prediction,
            "answerable": item["answerable"],
            "exact_match": exact,
            "token_f1": f1,
        }, ensure_ascii=False) + "\n")
        results_file.flush()

        line = "\n".join([
            "",
            "=" * 100,
            f"[{index}/{len(test_records)}]",
            f"Question : {item['question']}",
            f"Expected : {reference}",
            f"Generated: {prediction}",
            f"Exact/F1 : {exact} / {f1:.3f}",
        ])
        transcript_lines.append(line)
        print(line, flush=True)

total = len(test_records)
SUMMARY = [
    "",
    "=" * 100,
    "NO-CONTEXT RESULTS (raw base model, parametric knowledge only)",
    "=" * 100,
    f"Model      : {MODEL_ID}",
    f"Test file  : {TEST_PATH}",
    f"Exact match: {exact_correct}/{total} ({100 * exact_correct / total:.2f}%)",
    f"Mean token F1: {f1_total / total:.4f}",
    "=" * 100,
]
for line in SUMMARY:
    print(line)
transcript_lines.extend(SUMMARY)

with RESULTS_TXT.open("w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(transcript_lines) + "\n")

print(f"\nPer-item results : {RESULTS_JSONL}")
print(f"Transcript       : {RESULTS_TXT}")